# Visualizing regression results

In [1]:
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../../../updated-data/obs/lme-ready'
NULL_DATA_PATH = '../../../updated-data/null/null-lme-ready'

PARAMS_PATH = '../../3-REGRESSION-MODELS/reports/agg-params.csv'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
fs = [f for f in grab_all_files(DATA_PATH) if ('xling-' not in f)]
len(fs)

4518

## Importing parameters

In [5]:
dfps = pd.read_csv(PARAMS_PATH)
dfps['param'].loc[4] = 'null'
dfps.head()

,param,beta,sd,k,se,t,p
0,nx,0.161890,0.059459,12986,0.000522,310.269558,0.000000
1,ny,-0.026426,0.031582,12986,0.000277,-95.353114,0.000000
2,self,-0.194049,2.134731,12986,0.018733,-10.358721,0.000000
3,other,-0.036553,1.400310,12986,0.012288,-2.974622,0.001469
4,null,0.025137,1.037612,12986,0.009105,2.760681,0.002888


## Processing individual datasets

In [6]:
COEF_VAR = 'I'

In [7]:
# param_list = ['Intercept', 'nx', 'ny', 'self']
param_list = ['nx', 'ny', 'self']

In [8]:
VAL_COL = 'resid'
group_by_cols = ['corpus', 'turn_delta', 'self']

In [9]:
dataframe_cols = [
    'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
    COEF_VAR,
    'x_turn_id', 'y_turn_id', # 'AVAR',
    'nx', 'ny'
]

In [10]:
df_scale, df_regression = [], []

In [11]:
for f in tqdm(fs):

    dfo = pq.ParquetFile(os.path.join(DATA_PATH, f))
    if f.startswith('grace-'):
        dfn = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f.replace('grace', 'Miao-fNIRS')))
    else:
        dfn = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f))

    df = [dfo.read(columns=dataframe_cols).to_pandas()]
    df[-1]['null'] = False
    df += [dfn.read(columns=dataframe_cols).to_pandas()]
    df[-1]['null'] = True
    df = pd.concat(df, ignore_index=True)

    df = df.loc[
        (df['nx'] >= 5)
        & (df['ny'] >= 5)
    ]

    if ('CANDOR' in f.split('/')[-1]):
        df['corpus'] = 'CANDOR'

    if ('MICASE' in f.split('/')[-1]):
        df['corpus'] = 'MICASE'

    if ('callfriend' in f):
        df['corpus'] = 'callfriend'

    df['self'] = (df['x_speaker'] == df['y_speaker']).astype(int)
    df['self'].loc[df['null']] = 2

    # df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df['turn_delta'] = (df['y_turn_id'] - df['x_turn_id'])

    # if ('CANDOR' in f.split('/')[-1]) or ('grace' in f.split('/')[-1]):
    #     df['turn_delta'].loc[df['self'] == 0] -= 1

    if VAL_COL == 'resid':
        df['pred'] = (df['nx'] * dfps['beta'].loc[dfps['param'].isin(['nx'])].values[0]) + (df['ny'] * dfps['beta'].loc[dfps['param'].isin(['ny'])].values[0])

        df['resid'] = df[COEF_VAR] - df['pred']

    df_regression += [
        df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('mean').reset_index()
    ]

    df_regression[-1]['std'] = df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('std').reset_index()[VAL_COL]

    df_regression[-1]['k'] = df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('count').reset_index()[VAL_COL]

    df_regression[-1]['se'] = df_regression[-1]['std'] / np.sqrt(df_regression[-1]['k'])

df_regression = pd.concat(df_regression, ignore_index=True)

  0%|          | 0/4518 [00:00<?, ?it/s]

In [12]:
df0 = df_regression[['turn_delta', 'self'] + [VAL_COL]].groupby(by=['turn_delta', 'self']).agg('mean').reset_index()
df0.head()

,turn_delta,self,resid
0,1,0,-0.451458
1,1,1,-0.513141
2,1,2,-0.478856
3,2,0,-0.359128
4,2,1,-0.834568


In [13]:
df1 = df_regression[['turn_delta', 'self', 'corpus'] + [VAL_COL]].groupby(by=['turn_delta', 'self', 'corpus']).agg('mean').reset_index()
df1.head(n=25)

,turn_delta,self,corpus,resid
0,1,0,CABNC,-0.350540
1,1,0,CANDOR,-0.620169
2,1,0,CORAAL,-0.384065
3,1,0,DISPEL,-0.368291
4,1,0,Frederiksen,-0.223452
5,1,0,GCSAusE,-0.237337
6,1,0,Graesser,-0.373656
7,1,0,MICASE,-0.399615
8,1,0,SBCSAE,-0.241973
9,1,0,SCoSE,-0.369025


## Plotly vis

In [14]:
import plotly.graph_objects as go

### Aggregate/total visualization

Some of the corpora have different structures overall. This means that, for say CANDOR and the MIAO corpora, there is a strange distribution of turns such that self is always even turns away and other is always odd turns away. Especially because CANDOR is such a large component of the data, this caused a bumpier distribution than one would have expected.

To solve for this, we averaged odd versus even turns, smoothening out the overall visualization of the distribution.

##### Merge and average values

In [21]:
df01 = df0[['turn_delta', 'self', VAL_COL]].copy()
df01['turn_delta'] -= 1

In [22]:
df0 = pd.merge(
    left=df0, left_on=['turn_delta', 'self'],
    right=df01, right_on=['turn_delta', 'self'],
    how='left'
)
df0.head()

,turn_delta,self,resid_x,resid_y
0,1,0,-0.451458,-0.359128
1,1,1,-0.513141,-0.834568
2,1,2,-0.478856,-0.814301
3,2,0,-0.359128,-0.505037
4,2,1,-0.834568,-0.462420


In [23]:
df0['resid'] = df0[['resid_x', 'resid_y']].mean(axis=1)
df0.head()

,turn_delta,self,resid_x,resid_y,resid
0,1,0,-0.451458,-0.359128,-0.405293
1,1,1,-0.513141,-0.834568,-0.673854
2,1,2,-0.478856,-0.814301,-0.646579
3,2,0,-0.359128,-0.505037,-0.432082
4,2,1,-0.834568,-0.462420,-0.648494


### Main vis

In [15]:
# sel_1 = (df0['turn_delta'] <= 200) & (df0['turn_delta'] > 0) & ((df0['turn_delta'] % 2) != 0)
sel_1 = (df0['turn_delta'] <= 200) & (df0['turn_delta'] > 0) & ((df0['turn_delta'] % 2) == 0)

In [22]:
fig = go.Figure()

In [23]:
sel = df0['self'] == 1
fig.add_trace(
    go.Scatter(
        y = df0[VAL_COL].loc[sel & sel_1].values,
        # x = np.array(range(int((sel & sel_1).sum()))) + 1,
        x = df0['turn_delta'].loc[sel & sel_1].values,
        mode='lines',
        name='self',
        showlegend=True,
        legendgroup='self',
        line = dict(
            color='orange'
        )
    )
)

sel = df0['self'] == 0
fig.add_trace(
    go.Scatter(
        y = df0[VAL_COL].loc[sel & sel_1].values,
        # x = np.array(range(int((sel & sel_1).sum()))) + 1,
        x = df0['turn_delta'].loc[sel & sel_1].values,
        mode='lines',
        name='other',
        showlegend=True,
        legendgroup='other',
        line = dict(
            color='royalblue'
        )
    )
)

# sel = df0['self'] == 2
# fig.add_trace(
#     go.Scatter(
#         y = df0[VAL_COL].loc[sel & sel_1].values,
#         # x = np.array(range(int((sel & sel_1).sum()))) + 1,
#         x = df0['turn_delta'].loc[sel & sel_1].values,
#         mode='lines',
#         name='null',
#         showlegend=True,
#         legendgroup='null',
#         line = dict(
#             color='darkgray'
#         )
#     )
# )

fig.update_layout(
    template='xgridoff',
    yaxis_title='Residual I(x;y)',
    xaxis_title='Distance in turns from x to y'
)


In [24]:
fig.write_html(os.path.join(OUTPUTS_PATH, 'all-corpora.html'))
fig.write_image(os.path.join(OUTPUTS_PATH, 'all-corpora.png'), scale=5)

### Per corpus vis

In [19]:
figs = []

In [20]:
for corpus in df1.corpus.unique():
    sub = df1.loc[df1['corpus'].isin([corpus])]

    fig = go.Figure()

    sel_t = (sub['turn_delta'] > 0) & (sub['turn_delta'] <= 200)

    sel = sub['self'] == 1
    fig.add_trace(
        go.Scatter(
            x=sub['turn_delta'].loc[sel & sel_t].values,
            y = sub[VAL_COL].loc[sel & sel_t].values,
            mode='lines',
            name='self',
            showlegend=True,
            legendgroup='self',
            line = dict(
                color='orange'
            )
        )
    )

    sel = sub['self'] == 0
    fig.add_trace(
        go.Scatter(
            x=sub['turn_delta'].loc[sel & sel_t].values,
            y = sub[VAL_COL].loc[sel & sel_t].values,
            mode='lines',
            name='other',
            showlegend=True,
            legendgroup='other',
            line = dict(
                color='royalblue'
            )
        )
    )

    fig.update_layout(template='xgridoff')

    figs += [fig]

In [21]:
for corpus, plot in zip(df1.corpus.unique(), figs):
    print(corpus)
    plot.show()
    plot.write_html(os.path.join(OUTPUTS_PATH, corpus+'.html'))
    plot.write_image(os.path.join(OUTPUTS_PATH, corpus+'.png'), scale=5)

CABNC


CANDOR


CORAAL


DISPEL


Frederiksen


...

SCoSE


callfriend


callhome


grace


med_school
